# Held-Out Corner Data-Amount Sweep Predictions For Ansys

This notebook loads the saved models from the corner far-to-near data-amount sweep, selects a fixed 20-sample subset from the held-out corner test region, and writes one long CSV with the predicted designs for ten training percentages.

The exported CSV is the one to use for new Ansys simulations: 10 fractions x 20 held-out samples = 200 total rows. The old Ansys-validation target CSV is intentionally not used here because those samples were not guaranteed to come from this held-out corner test split.


In [1]:
from __future__ import annotations

import ast
import gc
import json
import os
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.models import load_model

try:
    tf.keras.backend.set_floatx("float32")
except Exception:
    pass

print("TensorFlow", tf.__version__)


TensorFlow 2.20.0


In [2]:
# Paths and run controls

HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon metadata file. Run from the repo root or the transmon experiment folder."
    )

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from parameters_surrogate_defined_loss import (  # noqa: E402
    EPOCHS,
    MODEL_DIR as PARAM_MODEL_DIR,
    SCALERS_DIR as PARAM_SCALERS_DIR,
    TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS,
)

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
RESULTS_DIR = EXPERIMENT_DIR / "results"
VALIDATION_DIR = RESULTS_DIR / "validation"
OUTPUT_DIR = VALIDATION_DIR / "data_amount_sweep_heldout_corner_ansys"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path(PARAM_MODEL_DIR)
SCALERS_DIR = Path(PARAM_SCALERS_DIR)

SURROGATE_MODEL_PATH = MODEL_DIR / "best_keras_model_model2_surrogate.keras"
SWEEP_MODEL_DIR = MODEL_DIR / "corner_far_to_near_data_amount_sweep_retrained_surrogate"
REFERENCE_COMBINED_MODEL_CANDIDATES = [
    MODEL_DIR / "surrogate_loss_2in_3out_best_model.keras",
    MODEL_DIR / "best_keras_model_surrogate_defined_loss.keras",
]
REFERENCE_COMBINED_MODEL_PATH = next((p for p in REFERENCE_COMBINED_MODEL_CANDIDATES if p.exists()), None)
if REFERENCE_COMBINED_MODEL_PATH is None:
    raise FileNotFoundError("No reference combined inverse+surrogate model found in MODEL_DIR")

CORNER_SWEEP_CSV = EXPERIMENT_DIR / "data_amount_sweep_corner_far_to_near_retrained_surrogate.csv"

# Run controls. Total exported Ansys rows =
# len(DEFAULT_FRACTIONS) x len(SEEDS_TO_RUN) x N_HELDOUT_TEST_SAMPLES_FOR_ANSYS.
# 10 fractions x 1 seed x 20 held-out samples = 200 rows.
SEEDS_TO_RUN = (0,)
DEFAULT_FRACTIONS = tuple(np.linspace(0.10, 1.00, 10).round(2))
N_HELDOUT_TEST_SAMPLES_FOR_ANSYS = 20
REQUIRE_SAVED_SWEEP_MODELS = True
DISABLE_JIT_COMPILE = False

TOTAL_ANSYS_ROWS = len(DEFAULT_FRACTIONS) * len(SEEDS_TO_RUN) * N_HELDOUT_TEST_SAMPLES_FOR_ANSYS

DETAIL_OUT_PATH = OUTPUT_DIR / "heldout_corner_test_predictions_all_fractions.csv"
ANSYS_INPUT_OUT_PATH = OUTPUT_DIR / "heldout_corner_test_ansys_input_all_fractions.csv"
SUMMARY_OUT_PATH = OUTPUT_DIR / "heldout_corner_test_surrogate_summary_by_model.csv"
HELDOUT_INDEX_OUT_PATH = OUTPUT_DIR / f"heldout_corner_test_metadata_indices_{TOTAL_ANSYS_ROWS}.csv"

print("Experiment:", EXPERIMENT_DIR)
print("Reference combined model:", REFERENCE_COMBINED_MODEL_PATH)
print("Saved sweep model dir:", SWEEP_MODEL_DIR)
print("Corner sweep CSV:", CORNER_SWEEP_CSV)
print("Output dir:", OUTPUT_DIR)


Experiment: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params
Reference combined model: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/surrogate_loss_2in_3out_best_model.keras
Saved sweep model dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate
Corner sweep CSV: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_corner_far_to_near_retrained_surrogate.csv
Output dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/validation/data_amount_sweep_heldout_corner_ansys


In [3]:
# Data loading, corner split recreation, and scaler helpers

EPS = 1e-12
HAMILTONIAN_COLUMN_NAMES = ["qubit_frequency_GHz", "anharmonicity_MHz"]
QISKIT_PARAM_NAMES = np.load(METADATA_DIR / "y_columns.npy", allow_pickle=True).astype(str).tolist()
DEFAULT_REQUESTED_TEST_CORNER_UM = np.array([50.0, 2.0, 420.0], dtype=np.float64)
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15

@dataclass
class SimpleMinMaxScaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (np.asarray(x, dtype=np.float64) - self.min_) / self.range_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return np.asarray(x, dtype=np.float64) * self.range_ + self.min_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


def load_metadata_arrays() -> tuple[np.ndarray, np.ndarray]:
    rows = json.loads(METADATA_PATH.read_text())
    hamiltonian = []
    geometry_um = []
    for row in rows:
        h = row["Hamiltonian_params"]
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]
        hamiltonian.append([
            float(h["qubit_frequency_GHz"]),
            float(h["anharmonicity_MHz"]),
        ])
        geometry_um.append([
            parse_um(readout["claw_length"]),
            parse_um(readout["ground_spacing"]),
            parse_um(opts["cross_length"]),
        ])
    return np.asarray(hamiltonian, dtype=np.float64), np.asarray(geometry_um, dtype=np.float64)


def requested_corner_from_sweep(default_um: np.ndarray) -> np.ndarray:
    if not CORNER_SWEEP_CSV.exists():
        return default_um
    sweep_head = pd.read_csv(CORNER_SWEEP_CSV, nrows=1)
    cols = ["requested_corner_claw_um", "requested_corner_ground_um", "requested_corner_cross_um"]
    if all(col in sweep_head.columns for col in cols):
        return sweep_head.loc[0, cols].to_numpy(dtype=np.float64)
    return default_um


def choose_corner_heldout_split(
    geometry_um: np.ndarray,
    requested_corner_um: np.ndarray,
    test_fraction: float = 0.15,
    val_fraction: float = 0.15,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n_rows = len(geometry_um)
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    geom_min = geometry_um.min(axis=0)
    geom_max = geometry_um.max(axis=0)
    clipped_corner_um = np.clip(requested_corner_um, geom_min, geom_max)

    corner_scaler = SimpleMinMaxScaler(geom_min, geom_max)
    geometry_scaled = corner_scaler.transform(geometry_um)
    corner_scaled = corner_scaler.transform(clipped_corner_um.reshape(1, -1))[0]

    corner_distance = np.linalg.norm(geometry_scaled - corner_scaled, axis=1)
    near_corner_order = np.argsort(corner_distance)

    corner_idx = near_corner_order[: n_test + n_val]
    rng = np.random.default_rng(42)
    rng.shuffle(corner_idx)

    test_idx = corner_idx[:n_test]
    val_idx = corner_idx[n_test:]
    train_pool_idx = near_corner_order[n_test + n_val :]
    return train_pool_idx, val_idx, test_idx, clipped_corner_um, corner_distance


def scaler_from_artifacts(values: np.ndarray, columns: list[str], path_patterns: list[str], label: str):
    mins, maxs, sources = [], [], []
    for i, col in enumerate(columns):
        loaded = None
        source = None
        for pattern in path_patterns:
            candidate = SCALERS_DIR / pattern.format(col=col)
            if candidate.exists():
                loaded = joblib.load(candidate)
                source = str(candidate)
                break
        if loaded is not None:
            mins.append(float(np.asarray(loaded.data_min_).reshape(-1)[0]))
            maxs.append(float(np.asarray(loaded.data_max_).reshape(-1)[0]))
            sources.append(source)
        else:
            mins.append(float(np.min(values[:, i])))
            maxs.append(float(np.max(values[:, i])))
            sources.append(f"metadata fallback: {label}.{col}")
    return SimpleMinMaxScaler(np.asarray(mins), np.asarray(maxs)), sources


h_raw, geom_raw_um = load_metadata_arrays()
geom_raw_si = geom_raw_um * 1e-6
all_idx = np.arange(len(h_raw))
REQUESTED_TEST_CORNER_UM = requested_corner_from_sweep(DEFAULT_REQUESTED_TEST_CORNER_UM)

train_pool_idx, val_idx, test_idx, clipped_corner_um, corner_distance = choose_corner_heldout_split(
    geom_raw_um,
    REQUESTED_TEST_CORNER_UM,
    TEST_FRACTION,
    VAL_FRACTION,
)

assert len(np.intersect1d(train_pool_idx, val_idx)) == 0
assert len(np.intersect1d(train_pool_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0

h_model_scaler, h_scaler_sources = scaler_from_artifacts(
    h_raw,
    HAMILTONIAN_COLUMN_NAMES,
    ["scaler_X_{col}.save", "scaler_X_linear_{col}.save"],
    "Hamiltonian",
)
geom_inverse_scaler, geom_inverse_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_{col}_one_hot_encoding.save"],
    "inverse_qiskit",
)
geom_surrogate_scaler, geom_surrogate_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_linear_{col}.save", "scaler_y_{col}_one_hot_encoding.save"],
    "surrogate_qiskit",
)

h_model_scaled = h_model_scaler.transform(h_raw).astype("float32")
geom_inverse_scaled = geom_inverse_scaler.transform(geom_raw_si).astype("float32")
geom_surrogate_scaled = geom_surrogate_scaler.transform(geom_raw_si).astype("float32")

# Converts inverse-output scaler space into surrogate-input scaler space.
scale_a = (geom_inverse_scaler.range_ / geom_surrogate_scaler.range_).astype("float32")
scale_b = ((geom_inverse_scaler.min_ - geom_surrogate_scaler.min_) / geom_surrogate_scaler.range_).astype("float32")

print(f"Total metadata rows: {len(h_raw)}")
print(f"Corner train pool / validation / test: {len(train_pool_idx)} / {len(val_idx)} / {len(test_idx)}")
print("Requested corner [um]:", REQUESTED_TEST_CORNER_UM)
print("Used clipped corner [um]:", clipped_corner_um)
print()
print("Split check passed:")
print("  train/val overlap:", len(np.intersect1d(train_pool_idx, val_idx)))
print("  train/test overlap:", len(np.intersect1d(train_pool_idx, test_idx)))
print("  val/test overlap:", len(np.intersect1d(val_idx, test_idx)))
print()
print("Scaler sources:")
all_scaler_sources = h_scaler_sources + geom_inverse_scaler_sources + geom_surrogate_scaler_sources
for name, source in zip(HAMILTONIAN_COLUMN_NAMES, h_scaler_sources):
    print(f"  H {name}: {source}")
for name, source in zip(QISKIT_PARAM_NAMES, geom_inverse_scaler_sources):
    print(f"  inverse geometry {name}: {source}")
for name, source in zip(QISKIT_PARAM_NAMES, geom_surrogate_scaler_sources):
    print(f"  surrogate geometry {name}: {source}")

if any(str(source).startswith("metadata fallback") for source in all_scaler_sources):
    print()
    print("WARNING: One or more saved scaler artifacts are missing, so this notebook is using metadata-derived fallback ranges.")
    print("The definitive comparison should be run with the original scalers restored in SCALERS_DIR.")


Total metadata rows: 1934
Corner train pool / validation / test: 1352 / 291 / 291
Requested corner [um]: [ 50.   2. 420.]
Used clipped corner [um]: [ 70.    4.1 420. ]

Split check passed:
  train/val overlap: 0
  train/test overlap: 0
  val/test overlap: 0

Scaler sources:
  H qubit_frequency_GHz: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_qubit_frequency_GHz.save
  H anharmonicity_MHz: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_anharmonicity_MHz.save
  inverse geometry design_options.connection_pads.readout.claw_length: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_y_design_options.connection_pads.readout.claw_length_one_hot_encoding.save
  inverse geometry design_options.connection_pads.readout.ground_spacing: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross

In [4]:
# Select the fixed evaluation set (N_HELDOUT_TEST_SAMPLES_FOR_ANSYS samples) from the held-out corner test region.

if len(test_idx) < N_HELDOUT_TEST_SAMPLES_FOR_ANSYS:
    raise ValueError(
        f"Requested {N_HELDOUT_TEST_SAMPLES_FOR_ANSYS} held-out test samples, "
        f"but the corner test split only has {len(test_idx)} samples."
    )

# test_idx is already the seeded random subset of the held-out corner. Taking the
# first 10 keeps the target set fixed across every fraction.
heldout_test_idx_for_ansys = np.asarray(test_idx[:N_HELDOUT_TEST_SAMPLES_FOR_ANSYS], dtype=int)
heldout_test_sample_ids = np.arange(len(heldout_test_idx_for_ansys), dtype=int)
heldout_target_h_raw = h_raw[heldout_test_idx_for_ansys]
heldout_target_h_scaled = h_model_scaled[heldout_test_idx_for_ansys]
heldout_target_geom_si = geom_raw_si[heldout_test_idx_for_ansys]
heldout_target_geom_um = geom_raw_um[heldout_test_idx_for_ansys]

assert len(np.intersect1d(heldout_test_idx_for_ansys, train_pool_idx)) == 0
assert len(np.intersect1d(heldout_test_idx_for_ansys, val_idx)) == 0

heldout_index_df = pd.DataFrame(
    {
        "test_sample_number": heldout_test_sample_ids,
        "metadata_index": heldout_test_idx_for_ansys,
        "target_qubit_frequency_GHz": heldout_target_h_raw[:, 0],
        "target_anharmonicity_MHz": heldout_target_h_raw[:, 1],
        "target_connection_pads.readout.claw_length_um": heldout_target_geom_um[:, 0],
        "target_connection_pads.readout.ground_spacing_um": heldout_target_geom_um[:, 1],
        "target_cross_length_um": heldout_target_geom_um[:, 2],
        "corner_distance": corner_distance[heldout_test_idx_for_ansys],
    }
)
heldout_index_df.to_csv(HELDOUT_INDEX_OUT_PATH, index=False)

print("Held-out corner Ansys target samples:", len(heldout_test_idx_for_ansys))
print("Metadata index CSV:", HELDOUT_INDEX_OUT_PATH)
heldout_index_df.head()


Held-out corner Ansys target samples: 10
Metadata index CSV: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/validation/data_amount_sweep_heldout_corner_ansys/heldout_corner_test_metadata_indices_100.csv


,test_sample_number,metadata_index,target_qubit_frequency_GHz,target_anharmonicity_MHz,target_connection_pads.readout.claw_length_um,target_connection_pads.readout.ground_spacing_um,target_cross_length_um,corner_distance
0,0,1348,4.033043,-144.390723,70.0,5.0,270.0,0.479459
1,1,1467,4.477979,-181.420265,70.0,5.0,220.0,0.624963
2,2,728,3.449059,-103.100450,170.0,4.1,380.0,0.326374
3,3,1233,3.596702,-112.787083,210.0,4.1,350.0,0.474317
4,4,1365,4.376285,-172.513248,150.0,5.0,230.0,0.643067


In [5]:
# Reference-model extraction and model builders


def read_keras_config(path: Path) -> dict:
    with zipfile.ZipFile(path) as zf:
        return json.loads(zf.read("config.json"))


def extract_reference_specs(combined_path: Path) -> tuple[dict, dict]:
    cfg = read_keras_config(combined_path)
    if cfg.get("class_name") == "Sequential" and cfg.get("config", {}).get("name") == "inverse_model":
        raise ValueError(
            f"{combined_path} appears to be the inverse-only model. Use the combined inverse+surrogate model."
        )
    layers = cfg["config"].get("layers", [])
    inverse_layer = next(
        layer for layer in layers
        if layer.get("class_name") == "Sequential" and layer.get("config", {}).get("name") == "inverse_model"
    )
    compile_cfg = cfg.get("compile_config") or {}
    return inverse_layer["config"], compile_cfg


def _initializer_from_config(config: dict | None, seed: int | None):
    if not config:
        return None
    cfg = json.loads(json.dumps(config))
    if seed is not None and isinstance(cfg.get("config"), dict) and "seed" in cfg["config"]:
        cfg["config"]["seed"] = seed
    try:
        return tf.keras.initializers.deserialize(cfg)
    except Exception:
        class_name = cfg.get("class_name")
        if class_name == "HeNormal":
            return tf.keras.initializers.HeNormal(seed=cfg.get("config", {}).get("seed"))
        if class_name == "LecunUniform":
            return tf.keras.initializers.LecunUniform(seed=cfg.get("config", {}).get("seed"))
        if class_name == "Zeros":
            return tf.keras.initializers.Zeros()
        return tf.keras.initializers.get(class_name)


def _regularizer_from_config(config: dict | None):
    if not config:
        return None
    try:
        return tf.keras.regularizers.deserialize(config)
    except Exception:
        if config.get("class_name") == "L2":
            return tf.keras.regularizers.l2(config.get("config", {}).get("l2", 0.01))
        raise


class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault("trainable", False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = {
            "scale_a": list(np.asarray(scale_a, dtype=float)),
            "scale_b": list(np.asarray(scale_b, dtype=float)),
        }

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config


def qiskit_range_penalty(y_true_dummy, y_pred):
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)


def build_inverse_from_reference(input_dim: int, seed: int) -> Sequential:
    tf.keras.utils.set_random_seed(seed)
    model = Sequential(name=REFERENCE_INVERSE_CONFIG.get("name", "inverse_model"))
    for layer_idx, layer_cfg in enumerate(REFERENCE_INVERSE_CONFIG["layers"]):
        class_name = layer_cfg["class_name"]
        cfg = layer_cfg["config"]
        if class_name == "InputLayer":
            model.add(Input(shape=(input_dim,), name=cfg.get("name", "Hamiltonian_input")))
        elif class_name == "Dense":
            model.add(
                Dense(
                    cfg["units"],
                    activation=cfg.get("activation", "linear"),
                    name=cfg.get("name"),
                    kernel_initializer=_initializer_from_config(cfg.get("kernel_initializer"), seed + layer_idx),
                    bias_initializer=_initializer_from_config(cfg.get("bias_initializer"), None),
                    kernel_regularizer=_regularizer_from_config(cfg.get("kernel_regularizer")),
                    bias_regularizer=_regularizer_from_config(cfg.get("bias_regularizer")),
                )
            )
        elif class_name == "LeakyReLU":
            model.add(LeakyReLU(negative_slope=cfg.get("negative_slope", 0.01), name=cfg.get("name")))
        elif class_name == "Dropout":
            model.add(Dropout(rate=cfg.get("rate", 0.0), name=cfg.get("name")))
        else:
            raise ValueError(f"Unsupported reference layer type: {class_name}")
    return model


def deserialize_loss_name(loss_spec):
    if isinstance(loss_spec, str):
        return loss_spec
    if isinstance(loss_spec, dict) and loss_spec.get("config") == "qiskit_range_penalty":
        return qiskit_range_penalty
    return loss_spec


def build_reference_optimizer():
    return tf.keras.optimizers.deserialize(REFERENCE_COMPILE_CONFIG["optimizer"])


def load_frozen_surrogate():
    surrogate_model = load_model(SURROGATE_MODEL_PATH, compile=False)
    surrogate_model.trainable = False
    for layer in surrogate_model.layers:
        layer.trainable = False
    return surrogate_model


def build_combined_model(seed: int) -> tuple[Sequential, Model]:
    tf.keras.backend.clear_session()
    surrogate_model = load_frozen_surrogate()
    inverse_model = build_inverse_from_reference(h_model_scaled.shape[1], seed=seed)

    combined_input = Input(shape=(h_model_scaled.shape[1],), name="combined_input")
    predicted_qiskit = inverse_model(combined_input)
    predicted_qiskit_converted = ScalerConversionLayer(scale_a, scale_b, name="scaler_conversion")(predicted_qiskit)
    reconstructed_hamiltonian = surrogate_model(predicted_qiskit_converted)
    combined_model = Model(
        inputs=combined_input,
        outputs=[reconstructed_hamiltonian, predicted_qiskit],
        name="combined_model",
    )

    loss_spec = REFERENCE_COMPILE_CONFIG.get("loss", [TRAIN_LOSS, qiskit_range_penalty])
    if not isinstance(loss_spec, list):
        loss_spec = [TRAIN_LOSS, qiskit_range_penalty]
    loss = [deserialize_loss_name(loss_spec[0]), qiskit_range_penalty]
    loss_weights = [float(v) for v in REFERENCE_COMPILE_CONFIG.get("loss_weights", [1.0, 1.0])]
    jit_compile = bool(REFERENCE_COMPILE_CONFIG.get("jit_compile", False)) and not DISABLE_JIT_COMPILE

    combined_model.compile(
        optimizer=build_reference_optimizer(),
        loss=loss,
        loss_weights=loss_weights,
        jit_compile=jit_compile,
    )
    return inverse_model, combined_model


REFERENCE_INVERSE_CONFIG, REFERENCE_COMPILE_CONFIG = extract_reference_specs(REFERENCE_COMBINED_MODEL_PATH)
reference_dense_layers = [
    layer["config"].get("units")
    for layer in REFERENCE_INVERSE_CONFIG["layers"]
    if layer["class_name"] == "Dense"
]

print("Reference inverse Dense units, including output layer:", reference_dense_layers)
print("Reference optimizer:", REFERENCE_COMPILE_CONFIG.get("optimizer", {}).get("class_name"))
print("Reference optimizer config:", REFERENCE_COMPILE_CONFIG.get("optimizer", {}).get("config"))
print("Reference loss:", REFERENCE_COMPILE_CONFIG.get("loss"))
print("Reference loss weights:", REFERENCE_COMPILE_CONFIG.get("loss_weights"))
print("Reference jit_compile:", REFERENCE_COMPILE_CONFIG.get("jit_compile"))


Reference inverse Dense units, including output layer: [64, 3]
Reference optimizer: Adam
Reference optimizer config: {'name': 'adam', 'learning_rate': {'module': 'keras.optimizers.schedules', 'class_name': 'ExponentialDecay', 'config': {'initial_learning_rate': 0.001, 'decay_steps': 220, 'decay_rate': 0.99, 'staircase': False, 'name': 'ExponentialDecay'}, 'registered_name': None}, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}
Reference loss: ['mae', {'module': 'builtins', 'class_name': 'function', 'config': 'qiskit_range_penalty', 'registered_name': 'function'}]
Reference loss weights: [1.0, 1]
Reference jit_compile: True


In [6]:
# Loading, evaluation, and CSV helpers

CUSTOM_OBJECTS = {
    "ScalerConversionLayer": ScalerConversionLayer,
    "qiskit_range_penalty": qiskit_range_penalty,
}


def saved_sweep_model_paths_for_seed(seed: int, fraction: float) -> tuple[Path, Path, Path]:
    pct = int(round(fraction * 100))
    stem = f"fraction_{pct:03d}pct_seed{seed}"
    return (
        SWEEP_MODEL_DIR / f"{stem}_surrogate.keras",
        SWEEP_MODEL_DIR / f"{stem}_combined.keras",
        SWEEP_MODEL_DIR / f"{stem}_inverse.keras",
    )


def load_sweep_model(seed: int, fraction: float) -> Model:
    _, combined_path, _ = saved_sweep_model_paths_for_seed(seed, fraction)
    if combined_path.exists():
        print(f"Loading {fraction:.0%} sweep model for seed {seed}: {combined_path}")
        return load_model(combined_path, compile=False, custom_objects=CUSTOM_OBJECTS)

    message = (
        f"Missing saved sweep model for fraction={fraction:.0%}, seed={seed}: {combined_path}. "
        "Run ml_32_corner_far_to_near.ipynb first so it saves every fraction/seed model."
    )
    if REQUIRE_SAVED_SWEEP_MODELS:
        raise FileNotFoundError(message)
    raise FileNotFoundError(message)


def predict_hamiltonian_targets(combined_model: Model, h_scaled: np.ndarray):
    pred_h_scaled, pred_geom_inverse_scaled = combined_model.predict(np.asarray(h_scaled, dtype="float32"), verbose=0)
    pred_h_raw = h_model_scaler.inverse_transform(pred_h_scaled)
    pred_geom_si = geom_inverse_scaler.inverse_transform(pred_geom_inverse_scaled)
    return pred_h_raw, pred_geom_si, pred_geom_inverse_scaled


def percent_errors(pred_h_raw: np.ndarray, target_h_raw: np.ndarray) -> np.ndarray:
    return 100.0 * np.abs(pred_h_raw - target_h_raw) / np.maximum(np.abs(target_h_raw), EPS)


def evaluate_target_set(
    combined_model: Model,
    h_scaled: np.ndarray,
    h_target_raw: np.ndarray,
    sample_ids: np.ndarray,
    metadata_indices: np.ndarray,
    seed: int,
    fraction: float,
) -> pd.DataFrame:
    pred_h_raw, pred_geom_si, pred_geom_scaled = predict_hamiltonian_targets(combined_model, h_scaled)
    pct = percent_errors(pred_h_raw, h_target_raw)
    pct_int = int(round(fraction * 100))
    pred_geom_um = pred_geom_si * 1e6

    return pd.DataFrame(
        {
            "ansys_job_id": [f"pct{pct_int:03d}_seed{seed}_test{sample_id:03d}" for sample_id in sample_ids],
            "fraction": fraction,
            "training_percent": fraction * 100.0,
            "seed": seed,
            "test_sample_number": sample_ids,
            "metadata_index": metadata_indices,
            "ref_qubit_frequency_GHz": h_target_raw[:, 0],
            "ref_anharmonicity_MHz": h_target_raw[:, 1],
            "ref_connection_pads.readout.claw_length": geom_raw_si[metadata_indices, 0],
            "ref_connection_pads.readout.ground_spacing": geom_raw_si[metadata_indices, 1],
            "ref_cross_length": geom_raw_si[metadata_indices, 2],
            "ref_connection_pads.readout.claw_length_um": geom_raw_um[metadata_indices, 0],
            "ref_connection_pads.readout.ground_spacing_um": geom_raw_um[metadata_indices, 1],
            "ref_cross_length_um": geom_raw_um[metadata_indices, 2],
            "pred_qubit_frequency_GHz": pred_h_raw[:, 0],
            "pred_anharmonicity_MHz": pred_h_raw[:, 1],
            "surrogate_frequency_pct_error": pct[:, 0],
            "surrogate_anharmonicity_pct_error": pct[:, 1],
            "surrogate_mean_hamiltonian_pct_error": pct.mean(axis=1),
            "pred_connection_pads.readout.claw_length": pred_geom_si[:, 0],
            "pred_connection_pads.readout.ground_spacing": pred_geom_si[:, 1],
            "pred_cross_length": pred_geom_si[:, 2],
            "pred_connection_pads.readout.claw_length_um": pred_geom_um[:, 0],
            "pred_connection_pads.readout.ground_spacing_um": pred_geom_um[:, 1],
            "pred_cross_length_um": pred_geom_um[:, 2],
            "pred_scaled_min": np.min(pred_geom_scaled, axis=1),
            "pred_scaled_max": np.max(pred_geom_scaled, axis=1),
        }
    )


ANSYS_INPUT_COLUMNS = [
    "ref_qubit_frequency_GHz",
    "ref_anharmonicity_MHz",
    "ref_connection_pads.readout.claw_length",
    "ref_connection_pads.readout.ground_spacing",
    "ref_cross_length",
    "pred_qubit_frequency_GHz",
    "pred_anharmonicity_MHz",
    "pred_connection_pads.readout.claw_length",
    "pred_connection_pads.readout.ground_spacing",
    "pred_cross_length",
]

ANSYS_RESULT_COLUMNS = [
    "ansys_qubit_frequency_GHz",
    "ansys_anharmonicity_MHz",
    "ansys_frequency_pct_error",
    "ansys_anharmonicity_pct_error",
    "ansys_mean_hamiltonian_pct_error",
]


def summarize_errors(label: str, df: pd.DataFrame, freq_col: str, alpha_col: str) -> dict:
    return {
        "set": label,
        "n": int(len(df)),
        "frequency_mean_pct": float(df[freq_col].mean()),
        "frequency_median_pct": float(df[freq_col].median()),
        "alpha_mean_pct": float(df[alpha_col].mean()),
        "alpha_median_pct": float(df[alpha_col].median()),
        "mean_hamiltonian_pct": float(df[[freq_col, alpha_col]].to_numpy().mean()),
    }


In [7]:
# Load every saved fraction/seed model and export held-out-corner predictions for Ansys.

if not CORNER_SWEEP_CSV.exists():
    raise FileNotFoundError(
        f"Missing retrained-surrogate sweep CSV: {CORNER_SWEEP_CSV}. "
        "Run ml_32_corner_far_to_near.ipynb first."
    )
if not SWEEP_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Missing retrained-surrogate sweep model directory: {SWEEP_MODEL_DIR}. "
        "Run ml_32_corner_far_to_near.ipynb first."
    )

reported = pd.read_csv(CORNER_SWEEP_CSV)
FRACTIONS_TO_RUN = DEFAULT_FRACTIONS
reported_fractions = np.asarray(sorted(float(v) for v in reported["fraction"].dropna().unique()), dtype=float)
missing_fractions = [fraction for fraction in FRACTIONS_TO_RUN if not np.isclose(reported_fractions, fraction).any()]
if missing_fractions:
    raise ValueError(
        "The retrained-surrogate sweep CSV does not contain the requested 10 fractions: "
        f"{missing_fractions}. Rerun ml_32_corner_far_to_near.ipynb after the fraction update."
    )


def sweep_row_for(fraction: float, seed: int) -> pd.Series | None:
    if reported.empty:
        return None
    mask = np.isclose(reported["fraction"].astype(float), fraction) & reported["seed"].astype(int).eq(seed)
    if not mask.any():
        return None
    return reported.loc[mask].iloc[0]


all_detail = []
summary_rows = []

for fraction in FRACTIONS_TO_RUN:
    for seed in SEEDS_TO_RUN:
        row = sweep_row_for(fraction, seed)
        surrogate_path, combined_path, inverse_path = saved_sweep_model_paths_for_seed(seed, fraction)
        combined_model = load_sweep_model(seed, fraction)

        detail = evaluate_target_set(
            combined_model,
            heldout_target_h_scaled,
            heldout_target_h_raw,
            heldout_test_sample_ids,
            heldout_test_idx_for_ansys,
            seed,
            fraction,
        )
        detail["target_set"] = f"heldout_corner_test_{N_HELDOUT_TEST_SAMPLES_FOR_ANSYS}_samples"
        detail["selection_method"] = f"fixed_{TOTAL_ANSYS_ROWS}_from_seeded_heldout_corner_test_split"
        detail["requested_corner_claw_um"] = REQUESTED_TEST_CORNER_UM[0]
        detail["requested_corner_ground_um"] = REQUESTED_TEST_CORNER_UM[1]
        detail["requested_corner_cross_um"] = REQUESTED_TEST_CORNER_UM[2]
        detail["used_corner_claw_um"] = clipped_corner_um[0]
        detail["used_corner_ground_um"] = clipped_corner_um[1]
        detail["used_corner_cross_um"] = clipped_corner_um[2]
        detail["corner_distance"] = corner_distance[heldout_test_idx_for_ansys]
        detail["n_samples"] = int(row["n_samples"]) if row is not None and "n_samples" in row else np.nan
        detail["saved_surrogate_model_path"] = str(row.get("saved_surrogate_model_path", surrogate_path)) if row is not None else str(surrogate_path)
        detail["saved_combined_model_path"] = str(row.get("saved_combined_model_path", combined_path)) if row is not None else str(combined_path)
        detail["saved_inverse_model_path"] = str(row.get("saved_inverse_model_path", inverse_path)) if row is not None else str(inverse_path)
        detail["source_sweep_csv"] = str(CORNER_SWEEP_CSV) if CORNER_SWEEP_CSV.exists() else ""
        for col in ANSYS_RESULT_COLUMNS:
            detail[col] = np.nan

        all_detail.append(detail)
        summary_rows.append(
            {
                "fraction": fraction,
                "training_percent": fraction * 100.0,
                "seed": seed,
                "n_samples": detail["n_samples"].iloc[0],
                "saved_combined_model_path": str(combined_path),
                **summarize_errors(
                    f"surrogate_only_on_fixed_heldout_corner_{TOTAL_ANSYS_ROWS}",
                    detail,
                    "surrogate_frequency_pct_error",
                    "surrogate_anharmonicity_pct_error",
                ),
            }
        )

        del combined_model
        tf.keras.backend.clear_session()
        gc.collect()

prediction_df = pd.concat(all_detail, ignore_index=True)

front_cols = [
    "ansys_job_id",
    "target_set",
    "fraction",
    "training_percent",
    "seed",
    "n_samples",
    "test_sample_number",
    "metadata_index",
    "selection_method",
]
meta_cols = [
    "requested_corner_claw_um",
    "requested_corner_ground_um",
    "requested_corner_cross_um",
    "used_corner_claw_um",
    "used_corner_ground_um",
    "used_corner_cross_um",
    "corner_distance",
]
readable_cols = [
    "ref_connection_pads.readout.claw_length_um",
    "ref_connection_pads.readout.ground_spacing_um",
    "ref_cross_length_um",
    "pred_connection_pads.readout.claw_length_um",
    "pred_connection_pads.readout.ground_spacing_um",
    "pred_cross_length_um",
    "surrogate_frequency_pct_error",
    "surrogate_anharmonicity_pct_error",
    "surrogate_mean_hamiltonian_pct_error",
    "pred_scaled_min",
    "pred_scaled_max",
]
model_cols = [
    "saved_surrogate_model_path",
    "saved_combined_model_path",
    "saved_inverse_model_path",
    "source_sweep_csv",
]
ordered_cols = front_cols + meta_cols + ANSYS_INPUT_COLUMNS + readable_cols + model_cols + ANSYS_RESULT_COLUMNS
prediction_df = prediction_df[ordered_cols]
prediction_df.to_csv(DETAIL_OUT_PATH, index=False)

ansys_input_df = prediction_df[front_cols + ANSYS_INPUT_COLUMNS + ANSYS_RESULT_COLUMNS].copy()
ansys_input_df.to_csv(ANSYS_INPUT_OUT_PATH, index=False)

summary_df = pd.DataFrame(summary_rows).sort_values(["training_percent", "seed"])
summary_df.to_csv(SUMMARY_OUT_PATH, index=False)

print(f"Wrote full prediction/detail CSV -> {DETAIL_OUT_PATH}")
print(f"Wrote Ansys input CSV with ids -> {ANSYS_INPUT_OUT_PATH}")
print(f"Wrote surrogate-only summary -> {SUMMARY_OUT_PATH}")
print("Rows to simulate:", len(ansys_input_df))
summary_df


Loading 10% sweep model for seed 0: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_010pct_seed0_combined.keras


I0000 00:00:1779900349.120873    1646 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1157 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 2g.20gb, pci bus id: 0000:00:10.0, compute capability: 8.0
I0000 00:00:1779900350.715481 1135500 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Loading 20% sweep model for seed 0: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_020pct_seed0_combined.keras
Loading 30% sweep model for seed 0: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_030pct_seed0_combined.keras
Loading 40% sweep model for seed 0: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_040pct_seed0_combined.keras
Loading 50% sweep model for seed 0: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_050pct_seed0_combined.keras
Loading 60% sweep model for seed 0: /home/olivias/ML_qubit_design/experiment

,fraction,training_percent,seed,n_samples,saved_combined_model_path,set,n,frequency_mean_pct,frequency_median_pct,alpha_mean_pct,alpha_median_pct,mean_hamiltonian_pct
0,0.1,10.0,0,135,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,13.655560,9.886421,1.964416,1.919882,7.809988
1,0.2,20.0,0,270,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.416494,0.431735,1.301310,1.381330,0.858902
2,0.3,30.0,0,406,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,32.643833,30.515287,6.546010,2.354274,19.594921
3,0.4,40.0,0,541,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.063494,0.020752,91.215453,82.984619,45.639474
4,0.5,50.0,0,676,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.134596,0.146747,0.170629,0.079551,0.152613
5,0.6,60.0,0,811,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.062095,0.063285,0.676375,0.319326,0.369235
6,0.7,70.0,0,946,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.203637,0.140388,0.734568,0.545721,0.469102
7,0.8,80.0,0,1082,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.176240,0.058691,0.672810,0.498349,0.424525
8,0.9,90.0,0,1217,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.127268,0.083757,0.474200,0.517089,0.300734
9,1.0,100.0,0,1352,/home/olivias/ML_qubit_design/experiments/mode...,surrogate_only_on_fixed_heldout_corner_100,10,0.103034,0.093674,0.789142,0.610979,0.446088


In [8]:
# Optional quick check: surrogate-only errors before running Ansys.
# The learning-curve plot in ml_34 will use the ansys_* columns after they are filled.

if DETAIL_OUT_PATH.exists():
    preview_df = pd.read_csv(DETAIL_OUT_PATH)
    preview_summary = (
        preview_df.groupby(["training_percent", "seed"], as_index=False)
        .agg(
            surrogate_frequency_mean_pct=("surrogate_frequency_pct_error", "mean"),
            surrogate_alpha_mean_pct=("surrogate_anharmonicity_pct_error", "mean"),
            surrogate_mean_hamiltonian_pct=("surrogate_mean_hamiltonian_pct_error", "mean"),
            n=("ansys_job_id", "size"),
        )
        .sort_values(["training_percent", "seed"])
    )
    display(preview_summary)
else:
    print("Run the export cell first.")


,training_percent,seed,surrogate_frequency_mean_pct,surrogate_alpha_mean_pct,surrogate_mean_hamiltonian_pct,n
0,10.0,0,13.655560,1.964416,7.809988,10
1,20.0,0,0.416494,1.301310,0.858902,10
2,30.0,0,32.643833,6.546010,19.594921,10
3,40.0,0,0.063494,91.215453,45.639474,10
4,50.0,0,0.134596,0.170629,0.152613,10
5,60.0,0,0.062095,0.676375,0.369235,10
6,70.0,0,0.203637,0.734568,0.469102,10
7,80.0,0,0.176240,0.672810,0.424525,10
8,90.0,0,0.127268,0.474200,0.300734,10
9,100.0,0,0.103034,0.789142,0.446088,10


## How To Read The Outputs

Use `results/validation/data_amount_sweep_heldout_corner_ansys/heldout_corner_test_ansys_input_all_fractions.csv` for the new Ansys simulations. It contains the same fixed 20 held-out corner test targets for each of ten training percentages, plus an `ansys_job_id` so the returned Ansys values can be merged back without ambiguity. This is 200 total Ansys rows.

The columns `ansys_qubit_frequency_GHz` and `ansys_anharmonicity_MHz` are intentionally blank at export time. After those are filled from Ansys, save the completed file as either `heldout_corner_test_predictions_with_ansys_results.csv` or overwrite the Ansys input CSV. `ml_34_corner_far_to_near_sweep_plots.ipynb` will then plot the data sweep using Ansys-simulated percent error against the target Hamiltonian values.

The full traceability file is `heldout_corner_test_predictions_all_fractions.csv`; it includes metadata indices, target/predicted geometries in meters and micrometers, surrogate-only errors, and saved model paths.
